In [34]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 36, Finished, Available, Finished, False)

In [35]:
lakehouse = "lh_airops360"

bronze_table = "bronze_faa_airport_operations"

silver_table = "silver_airport_capacity"

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 37, Finished, Available, Finished, False)

In [36]:
faa_df = spark.read.table(bronze_table)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 38, Finished, Available, Finished, False)

In [37]:
print(f"Total Rows : {faa_df.count()}")

display(faa_df.limit(10))

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 39, Finished, Available, Finished, False)

Total Rows : 16132


SynapseWidget(Synapse.DataFrame, 4afb1c1f-da95-4d47-979c-553cc4870767)

In [38]:
faa_df.printSchema()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 40, Finished, Available, Finished, False)

root
 |-- operation_date: date (nullable = true)
 |-- airport_code: string (nullable = true)
 |-- air_carrier_operations: long (nullable = true)
 |-- air_taxi_operations: long (nullable = true)
 |-- general_aviation_operations: long (nullable = true)
 |-- itinerant_military_operations: long (nullable = true)
 |-- itinerant_total_operations: long (nullable = true)
 |-- local_civil_operations: long (nullable = true)
 |-- local_military_operations: long (nullable = true)
 |-- local_total_operations: long (nullable = true)
 |-- total_operations: long (nullable = true)



In [39]:
display(faa_df)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7777190-1d98-484c-8f99-0706b5547d37)

In [40]:
print(f"Rows : {faa_df.count()}")

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 42, Finished, Available, Finished, False)

Rows : 16132


In [41]:
from pyspark.sql.functions import countDistinct

faa_df.select(countDistinct("airport_code")).show()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 43, Finished, Available, Finished, False)

+----------------------------+
|count(DISTINCT airport_code)|
+----------------------------+
|                         523|
+----------------------------+



In [42]:
from pyspark.sql.functions import min, max

faa_df.select(
    min("operation_date").alias("MinDate"),
    max("operation_date").alias("MaxDate")
).show()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 44, Finished, Available, Finished, False)

+----------+----------+
|   MinDate|   MaxDate|
+----------+----------+
|2024-01-01|2024-01-31|
+----------+----------+



In [43]:
from pyspark.sql.functions import col, sum

faa_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in faa_df.columns
]).show(vertical=True)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 45, Finished, Available, Finished, False)

-RECORD 0----------------------------
 operation_date                | 0   
 airport_code                  | 0   
 air_carrier_operations        | 0   
 air_taxi_operations           | 0   
 general_aviation_operations   | 0   
 itinerant_military_operations | 0   
 itinerant_total_operations    | 0   
 local_civil_operations        | 0   
 local_military_operations     | 0   
 local_total_operations        | 0   
 total_operations              | 0   



In [44]:
from pyspark.sql.functions import upper, trim

faa_df = faa_df.withColumn(
    "airport_code",
    upper(trim(col("airport_code")))
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 46, Finished, Available, Finished, False)

In [45]:
faa_df = faa_df.withColumn(
    "commercial_operations",
    col("air_carrier_operations") +
    col("air_taxi_operations")
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 47, Finished, Available, Finished, False)

In [46]:
faa_df = faa_df.withColumn(
    "military_operations",
    col("itinerant_military_operations") +
    col("local_military_operations")
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 48, Finished, Available, Finished, False)

In [47]:
faa_df = faa_df.withColumn(
    "civil_operations",
    col("general_aviation_operations") +
    col("local_civil_operations") +
    col("commercial_operations")
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 49, Finished, Available, Finished, False)

In [48]:
from pyspark.sql.functions import when

faa_df = faa_df.withColumn(
    "itinerant_pct",
    when(
        col("total_operations") > 0,
        col("itinerant_total_operations") / col("total_operations")
    )
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 50, Finished, Available, Finished, False)

In [49]:
faa_df = faa_df.withColumn(
    "local_pct",
    when(
        col("total_operations") > 0,
        col("local_total_operations") / col("total_operations")
    )
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 51, Finished, Available, Finished, False)

In [50]:
dim_airport = spark.read.table("dim_airport")

print(f"Rows : {dim_airport.count()}")

display(dim_airport.limit(10))

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 52, Finished, Available, Finished, False)

Rows : 6072


SynapseWidget(Synapse.DataFrame, 3cde3ab6-4b1a-43fb-ba6b-91992aea78fb)

In [51]:
dim_airport.printSchema()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 53, Finished, Available, Finished, False)

root
 |-- airport_key: integer (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- airport_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



In [52]:
dim_airport_lookup = (
    dim_airport
    .select(
        "airport_key",
        "iata_code"
    )
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 54, Finished, Available, Finished, False)

In [53]:
from pyspark.sql.functions import col

faa_df = (
    faa_df.alias("f")
    .join(
        dim_airport_lookup.alias("d"),
        col("f.airport_code") == col("d.iata_code"),
        "left"
    )
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 55, Finished, Available, Finished, False)

In [54]:
faa_df = faa_df.drop("iata_code")

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 56, Finished, Available, Finished, False)

In [55]:
faa_df.select(
    "airport_key",
    "airport_code"
).show(10)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 57, Finished, Available, Finished, False)

+-----------+------------+
|airport_key|airport_code|
+-----------+------------+
|       1442|         ERI|
|       5054|         TOP|
|        836|         CDW|
|       3480|         NEW|
|       NULL|         HYI|
|       1411|         EMT|
|       3086|         MDH|
|       NULL|         CXY|
|       3720|         OLM|
|       3834|         PAH|
+-----------+------------+
only showing top 10 rows



In [56]:
from pyspark.sql.functions import col

faa_df.filter(
    col("airport_key").isNull()
).count()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 58, Finished, Available, Finished, False)

1355

In [57]:
faa_df.filter(col("airport_key").isNull()).count()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 59, Finished, Available, Finished, False)

1355

In [58]:
from pyspark.sql.functions import col

(
    faa_df
    .filter(col("airport_key").isNull())
    .select("airport_code")
    .distinct()
    .orderBy("airport_code")
).show(1000, truncate=False)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 60, Finished, Available, Finished, False)

+------------+
|airport_code|
+------------+
|ASG         |
|BAK         |
|BKV         |
|CHD         |
|CPS         |
|CWF         |
|CXY         |
|DTO         |
|DTS         |
|EVB         |
|FDK         |
|FFZ         |
|FIN         |
|FWS         |
|GEU         |
|GKY         |
|GPM         |
|GSN         |
|GTU         |
|GYH         |
|HEF         |
|HKS         |
|HQZ         |
|HSA         |
|HUM         |
|HYI         |
|IFP         |
|JKA         |
|JQF         |
|JYO         |
|MIC         |
|MYF         |
|OMN         |
|OUN         |
|PMP         |
|RNM         |
|ROG         |
|RYY         |
|SGJ         |
|TKI         |
|UAO         |
|UES         |
|UNV         |
|WDG         |
+------------+



In [59]:
(
    faa_df
    .filter(col("airport_key").isNull())
    .select("airport_code")
    .distinct()
    .count()
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 61, Finished, Available, Finished, False)

44

In [60]:
(
    faa_df
    .filter(col("airport_key").isNull())
    .select("airport_code")
    .distinct()
    .orderBy("airport_code")
).show(50, truncate=False)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 62, Finished, Available, Finished, False)

+------------+
|airport_code|
+------------+
|ASG         |
|BAK         |
|BKV         |
|CHD         |
|CPS         |
|CWF         |
|CXY         |
|DTO         |
|DTS         |
|EVB         |
|FDK         |
|FFZ         |
|FIN         |
|FWS         |
|GEU         |
|GKY         |
|GPM         |
|GSN         |
|GTU         |
|GYH         |
|HEF         |
|HKS         |
|HQZ         |
|HSA         |
|HUM         |
|HYI         |
|IFP         |
|JKA         |
|JQF         |
|JYO         |
|MIC         |
|MYF         |
|OMN         |
|OUN         |
|PMP         |
|RNM         |
|ROG         |
|RYY         |
|SGJ         |
|TKI         |
|UAO         |
|UES         |
|UNV         |
|WDG         |
+------------+



In [61]:
from pyspark.sql.functions import col

faa_df = faa_df.filter(col("airport_key").isNotNull())

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 63, Finished, Available, Finished, False)

In [62]:
faa_df.filter(col("airport_key").isNull()).count()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 64, Finished, Available, Finished, False)

0

In [63]:
faa_df.count()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 65, Finished, Available, Finished, False)

14777

In [64]:
faa_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_faa_airport_capacity")

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 66, Finished, Available, Finished, False)

In [65]:
dim_date = spark.read.table("dim_date")

dim_date.printSchema()

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 67, Finished, Available, Finished, False)

root
 |-- flight_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- date_key: integer (nullable = true)



In [66]:
dim_date = spark.read.table("dim_date")

print(dim_date.count())

dim_date.printSchema()

display(dim_date.limit(10))

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 68, Finished, Available, Finished, False)

31
root
 |-- flight_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- date_key: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, 89d81acf-dee8-44c3-8fdf-be7d2c9791fe)

In [67]:
from pyspark.sql.functions import col

faa_df = (
    faa_df.alias("f")
    .join(
        dim_date.select("flight_date", "date_key").alias("d"),
        col("f.operation_date") == col("d.flight_date"),
        "left"
    )
    .drop("flight_date")
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 69, Finished, Available, Finished, False)

In [68]:
print("Rows :", faa_df.count())

print("Null Date Keys :",
      faa_df.filter(col("date_key").isNull()).count())

display(
    faa_df.select(
        "operation_date",
        "date_key",
        "airport_key"
    ).limit(10)
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 70, Finished, Available, Finished, False)

Rows : 14777
Null Date Keys : 0


SynapseWidget(Synapse.DataFrame, 64a6a521-0c42-4bbd-a94e-2a7fb56cf07c)

In [70]:

from pyspark.sql.functions import monotonically_increasing_id

fact_airport_capacity = (
    faa_df.select(
        "airport_key",
        "date_key",
        "air_carrier_operations",
        "air_taxi_operations",
        "general_aviation_operations",
        "itinerant_military_operations",
        "itinerant_total_operations",
        "local_civil_operations",
        "local_military_operations",
        "local_total_operations",
        "total_operations"
    )
    .withColumn(
        "airport_capacity_key",
        monotonically_increasing_id() + 1
    )
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 72, Finished, Available, Finished, False)

In [71]:
fact_airport_capacity = fact_airport_capacity.select(
    "airport_capacity_key",
    "date_key",
    "airport_key",
    "air_carrier_operations",
    "air_taxi_operations",
    "general_aviation_operations",
    "itinerant_military_operations",
    "itinerant_total_operations",
    "local_civil_operations",
    "local_military_operations",
    "local_total_operations",
    "total_operations"
)

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 73, Finished, Available, Finished, False)

In [72]:
print("Rows :", fact_airport_capacity.count())

fact_airport_capacity.printSchema()

display(fact_airport_capacity.limit(10))

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 74, Finished, Available, Finished, False)

Rows : 14777
root
 |-- airport_capacity_key: long (nullable = false)
 |-- date_key: integer (nullable = true)
 |-- airport_key: integer (nullable = true)
 |-- air_carrier_operations: long (nullable = true)
 |-- air_taxi_operations: long (nullable = true)
 |-- general_aviation_operations: long (nullable = true)
 |-- itinerant_military_operations: long (nullable = true)
 |-- itinerant_total_operations: long (nullable = true)
 |-- local_civil_operations: long (nullable = true)
 |-- local_military_operations: long (nullable = true)
 |-- local_total_operations: long (nullable = true)
 |-- total_operations: long (nullable = true)



SynapseWidget(Synapse.DataFrame, 7b57d608-6842-48e8-9d7c-615b3338eaa8)

In [75]:
fact_airport_capacity.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("fact_airport_capacity")

StatementMeta(, 5c9eedb6-8265-41c7-b0ca-9395d3aa4569, 77, Finished, Available, Finished, False)